# Matheel Examples in Colab

This notebook installs Matheel from the repository and walks through preprocessing, chunking, lexical scoring, dense embeddings, code metrics, archive ranking, and the comparison suite using `code_1.java` and `code_3_plag.java` from `sample_pairs.zip`.

If Colab shows an `HF_TOKEN` warning, you can ignore it for public models. The install cell is written to work in the current runtime without needing a restart.

The installation step can take a while because Colab may need to align preinstalled dependencies before Matheel is ready.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path


def run(*args, cwd=None):
    subprocess.check_call(list(args), cwd=cwd)


def ensure_colab_torch_stack():
    run(sys.executable, "-m", "pip", "install", "jedi>=0.16,<1")
    try:
        import torch
    except Exception as exc:
        print(f"Skipping torchvision/torchaudio pin because torch is unavailable: {exc}")
        return

    base_version, _, build_tag = torch.__version__.partition("+")
    compatibility = {
        "2.9.0": ("0.24.0", "2.9.0"),
        "2.9.1": ("0.24.1", "2.9.1"),
        "2.10.0": ("0.25.0", "2.10.0"),
    }
    versions = compatibility.get(base_version)
    if versions is None:
        print(f"Skipping torchvision/torchaudio pin for torch {torch.__version__}")
        return

    torchvision_version, torchaudio_version = versions
    index_urls = {
        "cpu": "https://download.pytorch.org/whl/cpu",
        "cu126": "https://download.pytorch.org/whl/cu126",
        "cu128": "https://download.pytorch.org/whl/cu128",
        "cu129": "https://download.pytorch.org/whl/cu129",
        "cu130": "https://download.pytorch.org/whl/cu130",
    }
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        f"torchvision=={torchvision_version}",
        f"torchaudio=={torchaudio_version}",
    ]
    index_url = index_urls.get(build_tag)
    if index_url:
        cmd.extend(["--index-url", index_url])
    run(*cmd)


repo_root = Path("/content/matheel")
if repo_root.exists():
    shutil.rmtree(repo_root)

run("git", "clone", "https://github.com/FahadEbrahim/matheel.git", str(repo_root))
os.chdir(repo_root)
ensure_colab_torch_stack()
run(sys.executable, "-m", "pip", "install", ".[all]")
print("Matheel is installed and ready in the current runtime. The dependency alignment step in Colab can take a few minutes.")

In [ ]:
from pathlib import Path
from zipfile import ZipFile

from matheel.chunking import chunk_text
from matheel.comparison_suite import run_comparison_suite
from matheel.preprocessing import preprocess_code
from matheel.similarity import calculate_similarity, get_sim_list

sample_archive = Path("/content/matheel/sample_pairs.zip")
code_a_name = "code_1.java"
code_b_name = "code_3_plag.java"

with ZipFile(sample_archive) as archive:
    code_a = archive.read(code_a_name).decode("utf-8")
    code_b = archive.read(code_b_name).decode("utf-8")

print(f"Loaded Code A: {code_a_name}")
print(f"Loaded Code B: {code_b_name}")

In [ ]:
print("=== Basic preprocessing ===")
print(preprocess_code(code_a, mode="basic"))
print()
print("=== Code chunks ===")
for index, chunk in enumerate(
    chunk_text(code_a, method="code", chunk_size=64, max_chunks=2, chunk_language="java"),
    start=1,
):
    print(f"Chunk {index}:")
    print(chunk)
    print()

In [ ]:
lexical_scores = {
    "levenshtein": calculate_similarity(code_a, code_b, feature_weights={"levenshtein": 1.0}),
    "jaro_winkler": calculate_similarity(code_a, code_b, feature_weights={"jaro_winkler": 1.0}),
    "winnowing": calculate_similarity(
        code_a,
        code_b,
        feature_weights={"winnowing": 1.0},
        winnowing_kgram=5,
        winnowing_window=4,
    ),
    "gst": calculate_similarity(
        code_a,
        code_b,
        feature_weights={"gst": 1.0},
        gst_min_match_length=5,
    ),
}

semantic_blend = calculate_similarity(
    code_a,
    code_b,
    model_name="huggingface/CodeBERTa-small-v1",
    feature_weights={"semantic": 0.7, "levenshtein": 0.3},
)

for name, score in lexical_scores.items():
    print(name, round(score, 4))
print("semantic_blend", round(semantic_blend, 4))

In [ ]:
code_metrics = {
    "codebleu": calculate_similarity(
        code_a,
        code_b,
        model_name="huggingface/CodeBERTa-small-v1",
        code_metric="codebleu",
        code_language="java",
        code_metric_weight=1.0,
        feature_weights={"code_metric": 1.0},
    ),
    "ruby": calculate_similarity(
        code_a,
        code_b,
        model_name="huggingface/CodeBERTa-small-v1",
        code_metric="ruby",
        code_language="java",
        code_metric_weight=1.0,
        feature_weights={"code_metric": 1.0},
    ),
    "tsed": calculate_similarity(
        code_a,
        code_b,
        model_name="huggingface/CodeBERTa-small-v1",
        code_metric="tsed",
        code_language="java",
        code_metric_weight=1.0,
        tsed_delete_cost=1.0,
        tsed_insert_cost=1.0,
        tsed_rename_cost=1.0,
        feature_weights={"code_metric": 1.0},
    ),
}

for name, score in code_metrics.items():
    print(name, round(score, 4))

In [ ]:
results = get_sim_list(
    sample_archive,
    model_name="huggingface/CodeBERTa-small-v1",
    threshold=0.2,
    number_results=10,
    feature_weights={"semantic": 0.7, "levenshtein": 0.3},
)
display(results.head().round(4))

runs = [
    {
        "run_name": "semantic_levenshtein_blend",
        "options": {
            "model_name": "huggingface/CodeBERTa-small-v1",
            "feature_weights": {"semantic": 0.7, "levenshtein": 0.3},
        },
    },
    {
        "run_name": "codebleu_java_blend",
        "options": {
            "model_name": "huggingface/CodeBERTa-small-v1",
            "code_metric": "codebleu",
            "code_language": "java",
            "code_metric_weight": 0.2,
            "feature_weights": {"semantic": 0.8, "code_metric": 0.2},
        },
    },
]

summary, details = run_comparison_suite(sample_archive, runs)
display(summary.round(4))
display(details["semantic_levenshtein_blend"].head().round(4))